### Day3. 로지스틱 회귀 (타이타닉 데이터셋)

타이타닉(titanic) 데이터를 사용해 로지스틱 회귀를 수행합니다.


In [ ]:
# 데이터 확인
import pandas as pd
pd.set_option('display.width', 120)
path = "https://raw.githubusercontent.com/Soyoung-Yoon/data_01/main/"
df = pd.read_csv(path + "titanic_03.csv")
# print(df.head(3))

   PassengerId  Survived  Pclass                                               Name  Gender   Age  SibSp  Parch  \
0            2         1       1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1      0   
1            4         1       1       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1      0   
2            5         0       3                           Allen, Mr. William Henry    male  35.0      0      0   

     Ticket     Fare Cabin Embarked  
0  PC 17599  71.2833   C85        C  
1    113803  53.1000  C123        S  
2    373450   8.0500   NaN        S  


다음과 같은 로지스틱회귀 모형을 사용한 분류모델을 만들고 결과를 확인합니다.
- titanic_03.csv 데이터를 사용합니다.
- 모델 생성시 상수항(=절편)을 포함하도록 하며, 규제는 사용하지 않습니다.
- 종속변수 : Survived
- 독립변수 : Pclass, Age, Fare, Gender (Gender만 범주형으로 사용합니다)
- Pclass와 Age 변수의 교호작용 효과 확인을 위한 항을 포함

In [5]:
# 3-1) 위의 조건에 따라 모델을 생성하고, summary()를 사용하여 결과를 확인합니다.
from statsmodels.api import GLM, families
formula = "Survived ~ Pclass * Age + Fare + C(Gender)"
model = GLM.from_formula(formula, df, family=families.Binomial()).fit()
print(model.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:               Survived   No. Observations:                  369
Model:                            GLM   Df Residuals:                      363
Model Family:                Binomial   Df Model:                            5
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -149.90
Date:                Sat, 15 Nov 2025   Deviance:                       299.80
Time:                        11:56:22   Pearson chi2:                     390.
No. Iterations:                     6   Pseudo R-squ. (CS):             0.4165
Covariance Type:            nonrobust                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept             5.1758      1.71

In [10]:
# 3-2) 분석 결과, Pclass와 Age의 교호작용 항에 대한 p-value를
# 반올림하여 소수점 아래 3자리까지 출력합니다.
print(model.pvalues['Pclass:Age'].round(3))

0.686


In [13]:
# 3-3) 위에서 생성된 model 결과를 기반으로, 다음 해석이 올바르면 1, 틀리면 0을 입력합니다.
# Pclass가 1단계 증가할 때 생존할 오즈는 감소한다.
y_1 = model.params['Pclass']
# print(y_1 / (1 - y_1)) # 1


In [15]:
# 3-4) 모델의 로그-우도를 반올림하여 소수점 아래 3자리까지 출력합니다.
print(model.llf.round(3))

-149.902


In [16]:
# 3-5) 모델의 잔차이탈도를 반올림하여 소수점 아래 3자리까지 출력합니다.
print(model.deviance.round(3))

299.804


In [17]:
# 3-6) 다음 sample의 P(Y=1)의 확률은? 반올림하여 소수점 아래 3자리까지 출력합니다.
sample = {'Age': [20], 'Fare': [50], 'Pclass': [2], 'Gender': ['female']}
data = pd.DataFrame(sample)
print(model.predict(data)[0].round(3))

0.926


In [19]:
# 3-7) 위 sample에 대한 로그-오즈는? 결과를 반올림하여 소수점 아래 4자리까지 출력한다.
print(model.predict(data, which='linear')[0].round(4))

2.5306


In [20]:
# 3-8) Pclass가 1단위 증가할 때 Survived 오즈가 약 몇% 감소하는가?
# 결과를 반올림하여 소수점 아래 2자리까지 출력한다.
odds_ratio = model.params['Pclass']
result = ((1 - odds_ratio) * 100).round(2)
print(result)

191.56


In [22]:
# 3-9) Gender가 'male'인 경우 Survived 오즈가 약 몇% 감소하는가?
# 결과를 반올림하여 소수점 아래 2자리까지 출력한다.
odds_ratio = model.params['C(Gender)[T.male]']
result = ((1 - odds_ratio) * 100).round(2)
print(result)

417.49


In [25]:
# 3-10) Fare가 10단위 증가할 때 Survived 오즈가 약 몇% 증가하는가?
# 결과를 반올림하여 소수점 아래 2자리까지 출력한다.
odds_ratio = model.params['Fare'] * 10
result = ((odds_ratio - 1) * 100).round(2)
print(abs(result))

95.84


In [26]:
# 3-11) 성별을 제외한 모든 조건이 동일할 때,
# 남성(male)의 생존 확률이 여성(female)보다 몇 %포인트 낮은가?
# 단, 확률 변화량은 (남성 생존 확률 - 여성 생존 확률)로 계산하며,
# 결과는 소수 둘째 자리까지 반올림하여 %p 단위로 표현하시오.

#본 문항은 '성별에 따른 생존 확률 변화량'을 구하는 것입니다.

# 표본 데이터 : Pclass=2, Age=30, Fare=50 인 남성/여성
sample = pd.DataFrame({'Pclass': [2, 2], 'Age': [30, 30], 'Fare': [50, 50], 'Gender': ['male', 'female']})


In [28]:
# 3-12) 모델의 회귀 계수합을 구하고, 반올림하여 소수점 아래 3자리까지 출력한다.
print(model.params.sum().round(3))

1.048
